In [1]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/katabatic1
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/katabatic1
 CONTRIBUTING.md		       main.py		        README.md
'C:\Users\Prabu\Downloads\Katabatic'   Makefile		        Results
 dev_deps.py			       medgan_adult.py	        runs
 discretized_data		       MODEL_CONTRIBUTIONS.md   sample_data
 encoded_data			       outputs		        scripts
 example.ipynb			       poetry.lock	        synthetic
 examples			       __pycache__	        utils.py
 katabatic			       pyproject.toml	        venv
 LICENSE			       raw_data


In [2]:
!ls katabatic/


cli  evaluate  __init__.py  models  pipeline  __pycache__  utils


In [6]:
%pip install -q numpy pandas scikit-learn scipy joblib tqdm seaborn matplotlib xgboost
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.tabkde.adapter import TabKDEAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "magic.csv"
SAMPLE_DIR = ROOT / "sample_data" / "magic"
SYNTH_DIR = ROOT / "synthetic" / "magic" / "tabkde"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COL = "class"

t0 = time.time()

print("Step 1: Train–Test Split")

pipeline = TrainTestSplitPipeline(
    model=lambda: TabKDEAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR),
    label_col=LABEL_COL
)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

encoder.fit(x_train_df.astype(str))

x_train_enc = encoder.transform(x_train_df.astype(str))
x_test_enc = encoder.transform(x_test_df.astype(str))

pd.DataFrame(x_train_enc).to_csv(SAMPLE_DIR / "x_train.csv", index=False)
pd.DataFrame(x_test_enc).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Step 2: TabKDE Training")

model = TabKDEAdapter()
model.train(output_dir=str(SAMPLE_DIR))

n_samples = len(x_train_df)

x_synth, y_synth = model.sample(n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({LABEL_COL: y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

y_test_df = pd.read_csv(SAMPLE_DIR / "y_test.csv")
y_test_df.columns = [LABEL_COL]
y_test_df.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

print("Step 3: TSTR Evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

runtime_min = round((time.time() - t0) / 60, 2)

print("RESULTS")
print(results)
print("Total runtime (minutes):", runtime_min)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Train–Test Split
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
Step 2: TabKDE Training
Step 3: TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/magic/tabkde_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.8260
F1 Score: 0.8225
AUC: 0.8695

MLP:
Accuracy: 0.8533
F1 Score: 0.8495
AUC: 0.9086

RF:
Accuracy: 0.8391
F1 Score: 0.8375
AUC: 0.8863

XGBoost:
Accuracy: 0.8462
F1 Score: 0.8466
AUC: 0.9088

================ RESULTS ================
{'LR': {'Accuracy': 0.8259726603575184, 'F1 Score': 0.8224645739069488, 'AUC': np.float64(0.8695438835123297)}, 'MLP': {'Accuracy': 0.8533123028391167, 'F1 Score': 0.8494897347533635, 'AUC': np.float64(0.9086103140225754)}, 'RF': {'Accuracy': 0.8391167192429022, 'F1 Score': 0.837457409037497, 'AUC': np.float64(0.8863485101415118)}, 'XGBoost': {'Accuracy': 0.8462145110410094, 'F1 Score': 0.8466087056866465, 'AUC': np.float64(0.908841863696042)}}
Total runtime (minutes): 0.23


In [7]:
import torch
import sklearn
import xgboost
print("All imports OK")


All imports OK


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.tabkde.adapter import TabKDEAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "adult.csv"
SAMPLE_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR = ROOT / "synthetic" / "adult" / "tabkde"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

print("Step 1: Train–Test Split")

pipeline = TrainTestSplitPipeline(
    model=lambda: TabKDEAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

y_train_df = pd.read_csv(SAMPLE_DIR / "y_train.csv")
LABEL_COL = y_train_df.columns[0]
print("Detected label column:", LABEL_COL)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str))
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str))
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Step 2: TabKDE Training")

model = TabKDEAdapter()
model.train(output_dir=str(SAMPLE_DIR))

n_samples = len(x_train_df)

x_synth, y_synth = model.sample(n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({LABEL_COL: y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

y_test_df = pd.read_csv(SAMPLE_DIR / "y_test.csv")
y_test_df.columns = [LABEL_COL]
y_test_df.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

print("Step 3: TSTR Evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

runtime_min = round((time.time() - t0) / 60, 2)

print("RESULTS")
print(results)
print("Total runtime (minutes):", runtime_min)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Train–Test Split
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Detected label column: class
Step 2: TabKDE Training
Step 3: TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/tabkde_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7837
F1 Score: 0.7614
AUC: 0.8116

MLP:
Accuracy: 0.8336
F1 Score: 0.8326
AUC: 0.8858

RF:
Accuracy: 0.8511
F1 Score: 0.8469
AUC: 0.8916

XGBoost:
Accuracy: 0.8310
F1 Score: 0.8384
AUC: 0.9167

================ RESULTS ================
{'LR': {'Accuracy': 0.7836634423460771, 'F1 Score': 0.761367959283922, 'AUC': np.float64(0.8115647118301316)}, 'MLP': {'Accuracy': 0.8335636419468755, 'F1 Score': 0.8325957410225606, 'AUC': np.float64(0.8858422881285982)}, 'RF': {'Accuracy': 0.8510670965760786, 'F1 Score': 0.8469066332809169, 'AUC': np.float64(0.8916055565299932)}, 'XGBoost': {'Accuracy': 0.8309534776600644, 'F1 Score': 0.8383565121713304, 'AUC': np.float64(0.9166668171313015)}}
Total runtime (minutes): 0.42


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.tabkde.adapter import TabKDEAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "shuttle.csv"
SAMPLE_DIR = ROOT / "sample_data" / "shuttle"
SYNTH_DIR = ROOT / "synthetic" / "shuttle" / "tabkde"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

print("Step 1: Train–Test Split")

pipeline = TrainTestSplitPipeline(
    model=lambda: TabKDEAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

y_train_df = pd.read_csv(SAMPLE_DIR / "y_train.csv")
LABEL_COL = y_train_df.columns[0]
print("Detected label column:", LABEL_COL)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str))
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str))
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Step 2: TabKDE Training")

model = TabKDEAdapter()
model.train(output_dir=str(SAMPLE_DIR))

n_samples = len(x_train_df)

x_synth, y_synth = model.sample(n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({LABEL_COL: y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

y_test_df = pd.read_csv(SAMPLE_DIR / "y_test.csv")
y_test_df.columns = [LABEL_COL]
y_test_df.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

print("Step 3: TSTR Evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

runtime_min = round((time.time() - t0) / 60, 2)

print("RESULTS")
print(results)
print("Total runtime (minutes):", runtime_min)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Train–Test Split
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
Detected label column: class
Step 2: TabKDE Training
Step 3: TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:44:36] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/shuttle/tabkde_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.9615
F1 Score: 0.9607

MLP:
Accuracy: 0.9987
F1 Score: 0.9986

RF:
Accuracy: 0.9992
F1 Score: 0.9991

XGBoost:
Accuracy: 0.9992
F1 Score: 0.9991

================ RESULTS ================
{'LR': {'Accuracy': 0.9614655172413793, 'F1 Score': 0.9607446834525789}, 'MLP': {'Accuracy': 0.9987068965517242, 'F1 Score': 0.9985548131003581}, 'RF': {'Accuracy': 0.9992241379310345, 'F1 Score': 0.9990949101989708}, 'XGBoost': {'Accuracy': 0.9992241379310345, 'F1 Score': 0.9991379593982347}}
Total runtime (minutes): 0.37


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.tabkde.adapter import TabKDEAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "car.csv"
SAMPLE_DIR = ROOT / "sample_data" / "car"
SYNTH_DIR = ROOT / "synthetic" / "car" / "tabkde"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

print("Step 1: Train–Test Split")

pipeline = TrainTestSplitPipeline(
    model=lambda: TabKDEAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

y_train_df = pd.read_csv(SAMPLE_DIR / "y_train.csv")
LABEL_COL = y_train_df.columns[0]
print("Detected label column:", LABEL_COL)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str))
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str))
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Step 2: TabKDE Training")

model = TabKDEAdapter()
model.train(output_dir=str(SAMPLE_DIR))

n_samples = len(x_train_df)

x_synth, y_synth = model.sample(n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({LABEL_COL: y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

y_test_df = pd.read_csv(SAMPLE_DIR / "y_test.csv")
y_test_df.columns = [LABEL_COL]
y_test_df.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

print("Step 3: TSTR Evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

runtime_min = round((time.time() - t0) / 60, 2)

print("RESULTS")
print(results)
print("Total runtime (minutes):", runtime_min)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Train–Test Split
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
Detected label column: 6
Step 2: TabKDE Training
Step 3: TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:47:15] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/car/tabkde_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7081
F1 Score: 0.6476

MLP:
Accuracy: 0.9422
F1 Score: 0.9402

RF:
Accuracy: 0.9393
F1 Score: 0.9378

XGBoost:
Accuracy: 0.9884
F1 Score: 0.9882

================ RESULTS ================
{'LR': {'Accuracy': 0.708092485549133, 'F1 Score': 0.6476386212802397}, 'MLP': {'Accuracy': 0.9421965317919075, 'F1 Score': 0.9402406910033757}, 'RF': {'Accuracy': 0.9393063583815029, 'F1 Score': 0.9378078174392017}, 'XGBoost': {'Accuracy': 0.9884393063583815, 'F1 Score': 0.9882294075544986}}
Total runtime (minutes): 0.13


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import time
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.tabkde.adapter import TabKDEAdapter
from katabatic.evaluate.tstr.evaluation import TSTREvaluation

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "nursery.csv"
SAMPLE_DIR = ROOT / "sample_data" / "nursery"
SYNTH_DIR = ROOT / "synthetic" / "nursery" / "tabkde"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

print("Step 1: Train–Test Split")

pipeline = TrainTestSplitPipeline(
    model=lambda: TabKDEAdapter()
)

pipeline.run(
    input_csv=str(RAW_CSV),
    output_dir=str(SAMPLE_DIR)
)

y_train_df = pd.read_csv(SAMPLE_DIR / "y_train.csv")
LABEL_COL = y_train_df.columns[0]
print("Detected label column:", LABEL_COL)

x_train_df = pd.read_csv(SAMPLE_DIR / "x_train.csv")
x_test_df = pd.read_csv(SAMPLE_DIR / "x_test.csv")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

encoder.fit(x_train_df.astype(str))

pd.DataFrame(
    encoder.transform(x_train_df.astype(str))
).to_csv(SAMPLE_DIR / "x_train.csv", index=False)

pd.DataFrame(
    encoder.transform(x_test_df.astype(str))
).to_csv(SAMPLE_DIR / "x_test.csv", index=False)

print("Step 2: TabKDE Training")

model = TabKDEAdapter()
model.train(output_dir=str(SAMPLE_DIR))

n_samples = len(x_train_df)

x_synth, y_synth = model.sample(n_samples)

pd.DataFrame(x_synth).to_csv(SYNTH_DIR / "x_synth.csv", index=False)
pd.DataFrame({LABEL_COL: y_synth}).to_csv(SYNTH_DIR / "y_synth.csv", index=False)

y_test_df = pd.read_csv(SAMPLE_DIR / "y_test.csv")
y_test_df.columns = [LABEL_COL]
y_test_df.to_csv(SAMPLE_DIR / "y_test.csv", index=False)

print("Step 3: TSTR Evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(SAMPLE_DIR)
)

results = tstr.evaluate()

runtime_min = round((time.time() - t0) / 60, 2)

print("RESULTS")
print(results)
print("Total runtime (minutes):", runtime_min)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Train–Test Split
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
Detected label column: 8
Step 2: TabKDE Training
Step 3: TSTR Evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:47:56] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/nursery/tabkde_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7616
F1 Score: 0.7545

MLP:
Accuracy: 0.9688
F1 Score: 0.9687

RF:
Accuracy: 0.9595
F1 Score: 0.9585

XGBoost:
Accuracy: 0.9992
F1 Score: 0.9992

================ RESULTS ================
{'LR': {'Accuracy': 0.7615740740740741, 'F1 Score': 0.75448143387201}, 'MLP': {'Accuracy': 0.96875, 'F1 Score': 0.9687025330739911}, 'RF': {'Accuracy': 0.9594907407407407, 'F1 Score': 0.958462558569607}, 'XGBoost': {'Accuracy': 0.9992283950617284, 'F1 Score': 0.9992229113984576}}
Total runtime (minutes): 0.25
